In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from factor_analyzer.factor_analyzer import (
    calculate_kmo,
    calculate_bartlett_sphericity,
)
from pyprojroot import here

In [2]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"
OUTPUT_DIR = here() / "results" / "validation"

In [3]:
df = pd.read_csv(PROCESSED_DATA_DIR/"RELMS_subindex.csv")

In [4]:
df["CVEGEO"] = df["CVEGEO"].astype(str).str.zfill(5)

In [5]:
idx_cols = [
    "idx_resources",
    "idx_energy",
    "idx_logistics",
    "idx_market",
    "idx_security",
]
 
X = df[idx_cols].copy()

### Although data already ranges between 0 and 1, it's unclear whether the indices follow the same distribution; therefore, we proceed to scale. 

In [6]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=idx_cols, index=df["CVEGEO"])

In [7]:
kmo_all, kmo_model = calculate_kmo(X_std)
chi_square_value, p_value = calculate_bartlett_sphericity(X_std)
 
print("=" * 60)
print("SAMPLING ADEQUACY CHECKS")
print("=" * 60)
print(f"KMO (overall measure of sampling adequacy): {kmo_model:.4f}")
print("  KMO per variable:")
for col, val in zip(idx_cols, kmo_all):
    print(f"    {col:<15s}: {val:.4f}")
print()
print(f"Bartlett's test of sphericity: chi2 = {chi_square_value:.4f}, "
      f"p-value = {p_value:.4e}")
if p_value < 0.05:
    print(" Reject H0: correlation matrix is NOT an identity matrix "
          "(PCA is appropriate).")
else:
    print(" Fail to reject H0: variables may be uncorrelated "
          "(PCA may not be meaningful).")
print()

SAMPLING ADEQUACY CHECKS
KMO (overall measure of sampling adequacy): 0.6120
  KMO per variable:
    idx_resources  : 0.2575
    idx_energy     : 0.8450
    idx_logistics  : 0.8565
    idx_market     : 0.5775
    idx_security   : 0.5763

Bartlett's test of sphericity: chi2 = 3588.3724, p-value = 0.0000e+00
 Reject H0: correlation matrix is NOT an identity matrix (PCA is appropriate).



In [8]:
corr_matrix = X.corr()
print("=" * 60)
print("PAIRWISE CORRELATION MATRIX")
print("=" * 60)
print(corr_matrix.round(4))
print()

PAIRWISE CORRELATION MATRIX
               idx_resources  idx_energy  idx_logistics  idx_market  \
idx_resources         1.0000     -0.0207         0.1518      0.0811   
idx_energy           -0.0207      1.0000         0.1390      0.3482   
idx_logistics         0.1518      0.1390         1.0000      0.3875   
idx_market            0.0811      0.3482         0.3875      1.0000   
idx_security          0.0375     -0.2945        -0.3574     -0.8143   

               idx_security  
idx_resources        0.0375  
idx_energy          -0.2945  
idx_logistics       -0.3574  
idx_market          -0.8143  
idx_security         1.0000  



In [9]:
pca = PCA()
scores = pca.fit_transform(X_std)
 
explained_var = pca.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)
 
print("=" * 60)
print("PCA EXPLAINED VARIANCE")
print("=" * 60)
for i, (ev, cum) in enumerate(zip(explained_var, cum_explained_var), start=1):
    print(f"  PC{i}: {ev:.4f}  (cumulative: {cum:.4f})")
print()
 
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loadings_df = pd.DataFrame(
    loadings,
    index=idx_cols,
    columns=[f"PC{i+1}" for i in range(loadings.shape[1])],
)
 
print("=" * 60)
print("PC1 / PC2 LOADINGS")
print("=" * 60)
print(loadings_df[["PC1", "PC2"]].round(4))
print()
 
same_sign_pc1 = np.all(loadings_df["PC1"] > 0) or np.all(loadings_df["PC1"] < 0)
print(f"All PC1 loadings same sign: {same_sign_pc1}")
if not same_sign_pc1:
    print("  WARNING: at least one index loads oppositely on PC1 -- "
          "this indicates the sub-indices may not form a single coherent "
          "latent dimension.")
print()

PCA EXPLAINED VARIANCE
  PC1: 0.4510  (cumulative: 0.4510)
  PC2: 0.2149  (cumulative: 0.6659)
  PC3: 0.1642  (cumulative: 0.8301)
  PC4: 0.1350  (cumulative: 0.9651)
  PC5: 0.0349  (cumulative: 1.0000)

PC1 / PC2 LOADINGS
                  PC1     PC2
idx_resources  0.0967  0.9035
idx_energy     0.5249 -0.2936
idx_logistics  0.6023  0.3854
idx_market     0.9106 -0.0352
idx_security  -0.8826  0.1511

All PC1 loadings same sign: False



In [10]:
fig, ax = plt.subplots(figsize=(7, 5))
components = np.arange(1, len(explained_var) + 1)
ax.plot(components, explained_var, "o-", linewidth=2, label="Explained variance")
ax.plot(components, cum_explained_var, "s--", linewidth=2,
        label="Cumulative explained variance")
ax.axhline(y=1 / len(idx_cols), color="gray", linestyle=":",
           label="Equal-contribution threshold")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title("Scree Plot - RELMS Sub-Indices PCA")
ax.set_xticks(components)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "scree_plot.png", dpi=300)
plt.close(fig)

In [12]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    loadings_df,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "Loading"},
)
ax.set_title("PCA Loadings - RELMS Sub-Indices")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "loadings_heatmap.png", dpi=300)
plt.close(fig)